# V7 — Output-Gated Segmented SSM + Graded Forget

V5 beat attention on quality (1.360 vs 1.388) but still trailed on speed
(1.7x) and peak VRAM (1.8x): the PyTorch Hillis-Steele scan writes
O(log T) intermediate (B,T,S) tensors to slow HBM.

V7 grafts two literature-backed mechanisms onto V6 (same Triton kernel,
same corpus, same protocol, budget locked EXACT):

- **Graft #1 — output gate + state normalization** (HGRN2/Mamba-2/Gated
  DeltaNet): `y_t = C·(o_t ⊙ RMSNorm(h_t))`, `o_t = σ(W_o x_t + b_o)`.
  Readout scale drifts as states accumulate; every strong post-Mamba block
  gates it. (HGRN2 ablation: removing the gate costs ~5.7% relative ppl.)
- **Graft #3 — layer-graded forget floor** (HGRN): per-layer learnable
  scalar added to the gate, init from σ⁻¹(0.3) → σ⁻¹(0.9) bottom→top, so
  low layers forget fast (spelling) and top layers retain (phrases).

The recurrence — and the fused Triton kernel — are UNCHANGED. Question:
do the two grafts push quality past V6's 1.364?


## 0. Setup (+ Triton)

In [1]:
import torch
import triton
print("torch", torch.__version__)
print("triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "V6 needs the T4 GPU"


torch 2.11.0+cu130
triton 3.6.0
cuda: True Tesla T4


## 1. Data (same 2.47 MB corpus) + boundary token id

In [2]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data)); train_data, val_data = data[:n], data[n:]
nl_id = stoi.get("\n")
print(f"corpus {len(text)/1e6:.2f} MB, vocab {vocab}, newline_id={nl_id}")
assert nl_id is not None, "newline not in vocab!"

SEQ, BS = 128, 32
def get_batch(split="train"):
    d = train_data if split == "train" else val_data
    i = torch.randint(0, len(d) - SEQ - 1, (BS,))
    x = torch.stack([d[j:j+SEQ] for j in i]).to(device)
    y = torch.stack([d[j+1:j+SEQ+1] for j in i]).to(device)
    return x, y


corpus 2.47 MB, vocab 104, newline_id=0


## 2. Models + Triton kernels (inlined, no imports)

In [3]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6: Triton-fused selective scan.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. KERNEL GATE — Triton vs V5 loop, delta < 1e-6 (else STOP)

In [4]:
torch.manual_seed(0)
B, T, S = 4, 128, 256
u = torch.randn(B, T, S, device=device)
gp = torch.randn(B, T, S, device=device)
keep = (torch.rand(B, T, device=device) > 0.3).float()

# Triton fused kernel (compiles on first call)
h_tri = triton_selective_scan(u, gp, keep)

# Ground truth: V5's original sequential loop semantics
g = torch.sigmoid(gp)
h = torch.zeros(B, S, device=device); hs = []
for t in range(T):
    k = keep[:, t:t+1]
    h = k * g[:, t] * h + k * (1 - g[:, t]) * u[:, t]
    hs.append(h)
h_loop = torch.stack(hs, 1)

# V5's Hillis-Steele scan path
a = keep.unsqueeze(-1) * g
b = keep.unsqueeze(-1) * (1 - g) * u
h_scan = parallel_scan_ab(a, b)

e_loop = float((h_tri - h_loop).abs().max())
e_scan = float((h_tri - h_scan).abs().max())
print(f"triton vs V5 loop max delta: {e_loop:.2e}")
print(f"triton vs V5 scan max delta: {e_scan:.2e}")

# backward: Triton grads vs autograd through the reference scan
u2 = u.clone().requires_grad_(True); gp2 = gp.clone().requires_grad_(True)
triton_selective_scan(u2, gp2, keep).pow(2).sum().backward()
u3 = u.clone().requires_grad_(True); gp3 = gp.clone().requires_grad_(True)
g3 = torch.sigmoid(gp3); a3 = keep.unsqueeze(-1) * g3
b3 = keep.unsqueeze(-1) * (1 - g3) * u3
parallel_scan_ab(a3, b3).pow(2).sum().backward()
du_err = float((u2.grad - u3.grad).abs().max())
dg_err = float((gp2.grad - gp3.grad).abs().max())
print(f"backward du max delta: {du_err:.2e}, dgp max delta: {dg_err:.2e}")
assert e_loop < 1e-6 and e_scan < 1e-6, "KERNEL GATE FAILED (forward)"
assert du_err < 1e-5 and dg_err < 1e-5, "KERNEL GATE FAILED (backward)"
print("KERNEL GATE PASSED — fused kernel == V5 math")


triton vs V5 loop max delta: 4.47e-07
triton vs V5 scan max delta: 4.77e-07
backward du max delta: 1.91e-06, dgp max delta: 1.67e-06
KERNEL GATE PASSED — fused kernel == V5 math


## 4. Build + verify parameter budget (identical to V5)

In [5]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

DIM, LAYERS, HEADS = 256, 8, 8
attention = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ,
    mixer_fn=lambda d, h: CausalSelfAttention(d, h)).to(device)
v7 = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ,
    mixer_fn=lambda d, h: SelectiveSegmentedStateV7(d, state_dim=256),
    ffn_hidden=1023, newline_id=nl_id).to(device)
import math as _math
_floors = torch.linspace(_math.log(0.3/0.7), _math.log(0.9/0.1), LAYERS)
for _i, _blk in enumerate(v7.blocks):
    _blk.mix.forget_floor.data.fill_(_floors[_i])
print("forget floors:", [f"{f:.2f}" for f in _floors.tolist()])
models = {"attention": attention, "v7-gated": v7}
for name, m in models.items():
    print(f"{name}: {count_params(m)} params")
pa, pv = count_params(attention), count_params(v7)
print(f"gap: {pa - pv} params ({100*(pa-pv)/pa:.4f}%)")
print(f"v7 state dim frozen at: {v7.blocks[0].mix.state_dim}")
assert abs(pa - pv) < 2000, "budget not locked!"
print("BUDGET GATE PASSED")


forget floors: ['-0.85', '-0.41', '0.02', '0.46', '0.89', '1.33', '1.76', '2.20']
attention: 6369792 params
v7-gated: 6369792 params
gap: 0 params (0.0000%)
v7 state dim frozen at: 256
BUDGET GATE PASSED


## 5. Train — identical protocol to V5 (1500 steps)

In [6]:
import math

STEPS, EVAL, LR = 1500, 250, 3e-4
opt = {n: torch.optim.AdamW(m.parameters(), lr=LR) for n, m in models.items()}
val_hist = {n: [] for n in models}

def val_loss(m):
    m.eval()
    with torch.no_grad():
        return sum(float(m(*get_batch("val"))[1]) for _ in range(10)) / 10

for step in range(1, STEPS + 1):
    for name, m in models.items():
        m.train()
        xb, yb = get_batch("train")
        _, loss = m(xb, yb)
        opt[name].zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt[name].step()
    if step % EVAL == 0 or step == 1:
        msg = f"step {step:5d}"
        for name, m in models.items():
            vl = val_loss(m)
            val_hist[name].append(vl)
            msg += f" | {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})"
        print(msg, flush=True)
print("V6 TRAINING DONE")

names = list(models)
print("leader per checkpoint:")
for i, s in enumerate([1] + list(range(EVAL, STEPS + 1, EVAL))):
    a, b = val_hist[names[0]][i], val_hist[names[1]][i]
    lead = names[0] if a < b else names[1]
    print(f"  step {s:5d}: {lead} leads ({a:.3f} vs {b:.3f})")
for name, m in models.items():
    vl = val_loss(m)
    print(f"FINAL {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})")
final_vals = {name: val_loss(m) for name, m in models.items()}


step     1 | attention: val 4.004 (ppl 54.8) | v7-gated: val 4.021 (ppl 55.8)
step   250 | attention: val 2.373 (ppl 10.7) | v7-gated: val 1.863 (ppl 6.4)
step   500 | attention: val 2.004 (ppl 7.4) | v7-gated: val 1.600 (ppl 5.0)
step   750 | attention: val 1.749 (ppl 5.7) | v7-gated: val 1.484 (ppl 4.4)
step  1000 | attention: val 1.572 (ppl 4.8) | v7-gated: val 1.405 (ppl 4.1)
step  1250 | attention: val 1.473 (ppl 4.4) | v7-gated: val 1.337 (ppl 3.8)
step  1500 | attention: val 1.413 (ppl 4.1) | v7-gated: val 1.334 (ppl 3.8)
V6 TRAINING DONE
leader per checkpoint:
  step     1: attention leads (4.004 vs 4.021)
  step   250: v7-gated leads (2.373 vs 1.863)
  step   500: v7-gated leads (2.004 vs 1.600)
  step   750: v7-gated leads (1.749 vs 1.484)
  step  1000: v7-gated leads (1.572 vs 1.405)
  step  1250: v7-gated leads (1.473 vs 1.337)
  step  1500: v7-gated leads (1.413 vs 1.334)
FINAL attention: val 1.423 (ppl 4.2)
FINAL v7-gated: val 1.339 (ppl 3.8)


## 6. Post-train metrics (runs immediately — same session, no idle gap)

In [7]:
import time

print("=== POST-TRAIN METRICS ===")
for name, m in models.items():
    m.train()
    tl = 0.0
    for _ in range(20):
        xb, yb = get_batch("train")
        with torch.no_grad():
            tl += float(m(xb, yb)[1])
    tl /= 20
    opt_m = torch.optim.AdamW(m.parameters(), lr=3e-4)
    xb, yb = get_batch("train")
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(30):
        _, loss = m(xb, yb)
        opt_m.zero_grad(); loss.backward(); opt_m.step()
    torch.cuda.synchronize()
    train_ms = (time.time()-t0)/30*1000
    train_tps = BS*SEQ/(train_ms/1000)
    torch.cuda.reset_peak_memory_stats()
    _, loss = m(xb, yb); opt_m.zero_grad(); loss.backward(); opt_m.step()
    peak_mb = torch.cuda.max_memory_allocated()/1e6
    torch.cuda.empty_cache()
    m.eval()
    with torch.no_grad():
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(100):
            m(xb)
        torch.cuda.synchronize()
    infer_ms = (time.time()-t0)/100*1000
    infer_tps = BS*SEQ/(infer_ms/1000)
    print(f"{name}: train_loss(pt)={tl:.3f} train_ms={train_ms:.1f} "
          f"train_tok/s={train_tps:.0f} peak_MB={peak_mb:.0f} "
          f"infer_ms={infer_ms:.1f} infer_tok/s={infer_tps:.0f}")
print("METRICS DONE")


=== POST-TRAIN METRICS ===
attention: train_loss(pt)=1.472 train_ms=64.3 train_tok/s=63749 peak_MB=835 infer_ms=20.8 infer_tok/s=197285
v7-gated: train_loss(pt)=1.392 train_ms=64.7 train_tok/s=63351 peak_MB=935 infer_ms=21.8 infer_tok/s=188101
METRICS DONE


## 7. Kernel benchmark — the software tax, measured (T=128/512/1024)

In [8]:
import time
print("=== KERNEL BENCHMARK: mixer-level fwd+bwd, batch 8, dim 256 ===")

def bench(make, T, iters=30):
    m = make().train().to(device)
    x = torch.randn(8, T, 256, device=device)
    r = (torch.rand(8, T, device=device) < 0.3).float()
    for _ in range(5):                      # warmup (Triton compiles here)
        m.zero_grad(); m(x, r).sum().backward()
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(iters):
        m.zero_grad(); m(x, r).sum().backward()
    torch.cuda.synchronize()
    ms = (time.time() - t0) / iters * 1000
    torch.cuda.reset_peak_memory_stats()
    m.zero_grad(); m(x, r).sum().backward()
    peak = torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.empty_cache()
    return ms, peak

mixers = {
    "attention      ": lambda: CausalSelfAttention(256, 8),
    "v5 scan (torch)": lambda: SelectiveSegmentedState(256, 256),
    "v6 triton      ": lambda: SelectiveSegmentedStateTriton(256, 256),
}
for T in (128, 512, 1024):
    print(f"--- T={T} ---")
    for name, make in mixers.items():
        ms, peak = bench(make, T)
        print(f"  {name}: {ms:7.2f} ms/fwd+bwd | peak {peak:6.0f} MB")
print("BENCHMARK DONE")


=== KERNEL BENCHMARK: mixer-level fwd+bwd, batch 8, dim 256 ===
--- T=128 ---
  attention      :    1.04 ms/fwd+bwd | peak    296 MB
  v5 scan (torch):    3.73 ms/fwd+bwd | peak    316 MB
  v6 triton      :    1.35 ms/fwd+bwd | peak    294 MB
--- T=512 ---
  attention      :    4.50 ms/fwd+bwd | peak    331 MB
  v5 scan (torch):   10.66 ms/fwd+bwd | peak    439 MB
  v6 triton      :    1.63 ms/fwd+bwd | peak    326 MB
--- T=1024 ---
  attention      :   13.52 ms/fwd+bwd | peak    377 MB
  v5 scan (torch):   22.76 ms/fwd+bwd | peak    619 MB
  v6 triton      :    3.31 ms/fwd+bwd | peak    368 MB
BENCHMARK DONE


## 8. Results file (backup record — uses stored FINAL vals, no re-eval)

In [9]:

lines = ["EQUALITY: attention=6369792 v7=6369792"]
import math
for name, vl in final_vals.items():
    lines.append(f"FINAL {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})")
open("/content/v7_results.txt", "w").write("\n".join(lines))
print(open("/content/v7_results.txt").read())
print("RESULTS FILE WRITTEN")


EQUALITY: attention=6369792 v7=6369792
FINAL attention: val 1.406 (ppl 4.1)
FINAL v7-gated: val 1.350 (ppl 3.9)
RESULTS FILE WRITTEN


## Reading V7

- **Kernel gate** must pass first: fused Triton == V5 math (< 1e-4).
- **Quality:** v6 should land where V5 did (~1.36) — same math, same crown.
- **Speed/VRAM:** the question — does Step A/B/C fusion close the 1.7x
  throughput and 1.8x VRAM gaps vs attention?
- A clean sweep (quality + speed + memory) would make the fused gated
  segmented state the strictly dominant architecture at this scale.
